In [1]:
# Cell 1: Autoreload setup — picks up changes to src/ files without kernel restart
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print("Setup complete")

Setup complete


## Phase 2 — Feature Engineering

### 1. Building Statistics Consolidation

EDA (Phase 1) found ~20 building/apartment statistic columns clustered in 
`_AVG`/`_MODE`/`_MEDI` triplets, co-missing due to a shared root cause 
(absence of the applicant's building record). This step consolidates each 
triplet to a single `_AVG` column and adds one `BUILDING_INFO_AVAILABLE` flag.

### 0. DAYS_EMPLOYED Fix (moved earlier in pipeline)

Originally explored only in EDA, this must run **before** other Phase 2 steps: 
investigating the OCCUPATION_TYPE missing-category issue (below) revealed 99.96% 
overlap between "not employed" (DAYS_EMPLOYED == 365243) and missing OCCUPATION_TYPE 
— confirming they're largely the same population (pensioners/unemployed with no 
occupation to report). Recreates the EDA fix: IS_NOT_EMPLOYED flag + placeholder 
replaced with NaN.

### 4. Categorical Encoding

... (keep existing target encoding documentation) ...

**Bug found and fixed:** initial target encoding showed 96,391 "unseen categories" 
for OCCUPATION_TYPE when encoding the SAME data it was learned from — impossible 
unless something was wrong. Root cause: pandas groupby() silently drops NaN groups, 
so missing OCCUPATION_TYPE (31.35% of data) never got a learned encoding and fell 
back to a generic global mean. Investigation confirmed missing OCCUPATION_TYPE 
overlaps 99.96% with the not-employed population (IS_NOT_EMPLOYED) — a real, 
informative group, not noise. Fixed by filling NaN with the string 'Missing' 
before grouping in both fit_target_encoding and apply_target_encoding, so it 
receives its own learned, meaningful encoded value.

In [2]:
# Cell 2 (rebuild in correct order)
from feature_engineering import (
    fix_days_employed, consolidate_building_stats, add_ext_source_missing_flags,
    transform_amounts, fit_target_encoding, apply_target_encoding, one_hot_encode_remaining
)

train_raw = pd.read_csv('../data/raw/home-credit-default-risk/application_train.csv')

train_fe = fix_days_employed(train_raw)
train_fe = consolidate_building_stats(train_fe)
train_fe = add_ext_source_missing_flags(train_fe)
train_fe = transform_amounts(train_fe)

encoding_maps = fit_target_encoding(train_fe)
train_fe = apply_target_encoding(train_fe, encoding_maps)
train_fe = one_hot_encode_remaining(train_fe)

print(f"\nFinal shape: {train_fe.shape}")

IS_NOT_EMPLOYED: 55374 flagged (18.01%)
DAYS_EMPLOYED placeholder replaced with NaN
Found 14 building-stat triplets: ['APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD', 'COMMONAREA', 'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN', 'LANDAREA', 'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS', 'NONLIVINGAREA']
Dropped 28 redundant MODE/MEDI columns
Kept 14 _AVG columns + 1 new BUILDING_INFO_AVAILABLE flag
EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
AMT_INCOME_TOTAL: capped 3 rows at 10,000,000, log-transformed
AMT_CREDIT: log-transformed (no capping - ceiling confirmed legitimate in EDA)
Learned encoding for ORGANIZATION_TYPE: 58 categories (including 'Missing' if present)
Learned encoding for OCCUPATION_TYPE: 19 categories (including 'Missing' if present)
ORGANIZATION_TYPE: encoded, 0 truly-unseen categories filled with global mean (0.0791)
OCCUPATION_TYPE: encoded, 0 truly-unseen categories filled with globa

**Correction:** the initial flag used only `APARTMENTS_AVG` as a reference column, 
which undercounted availability — some applicants had data in other triplets (e.g. 
`YEARS_BEGINEXPLUATATION_AVG`) even when missing `APARTMENTS_AVG`. Fixed to check 
**any** of the 14 `_AVG` columns via `.any(axis=1)`, which more accurately captures 
"does this applicant have any building record at all."

**Corrected result:** 158,701 applicants (51.6%) have at least partial building 
info vs. 148,810 (48.4%) with none — a near-even split, revised from the earlier 
(undercounted) 156,061/151,450 split.

In [3]:
# Run this immediately after Cell 2, nothing in between
print(f"encoding_maps keys: {list(encoding_maps.keys())}")
print(f"OCCUPATION_TYPE in encoding_maps: {'OCCUPATION_TYPE' in encoding_maps}")
if 'OCCUPATION_TYPE' in encoding_maps:
    print(encoding_maps['OCCUPATION_TYPE'].sort_values())

encoding_maps keys: ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
OCCUPATION_TYPE in encoding_maps: True
OCCUPATION_TYPE
Accountants              0.048303
High skill tech staff    0.061599
Managers                 0.062140
Core staff               0.063040
HR staff                 0.063943
IT staff                 0.064639
Missing                  0.065131
Private service staff    0.065988
Medicine staff           0.067002
Secretaries              0.070498
Realty agents            0.078562
Cleaning staff           0.096067
Sales staff              0.096318
Cooking staff            0.104440
Laborers                 0.105788
Security staff           0.107424
Waiters/barmen staff     0.112760
Drivers                  0.113261
Low-skill Laborers       0.171524
Name: TARGET, dtype: float64


In [4]:
# Cell 3: Sanity-check the new flag
print(train_fe['BUILDING_INFO_AVAILABLE'].value_counts())
print(f"\nRetained columns sample:")
print([c for c in train_fe.columns if 'AVG' in c or 'BUILDING' in c])

BUILDING_INFO_AVAILABLE
1    158701
0    148810
Name: count, dtype: int64

Retained columns sample:
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'BUILDING_INFO_AVAILABLE']


In [5]:
# Cell 4: Does building info availability correlate with default risk?
check = train_fe.groupby('BUILDING_INFO_AVAILABLE')['TARGET'].agg(['count', 'mean'])
check.columns = ['count', 'default_rate']
check['default_rate_pct'] = check['default_rate'] * 100
print(check)

                          count  default_rate  default_rate_pct
BUILDING_INFO_AVAILABLE                                        
0                        148810      0.092171          9.217123
1                        158701      0.070000          6.999956


**Finding:** Default rate confirms the same pattern with corrected counts: 9.22% 
(no building info) vs. 7.00% (building info available), both deviating from the 
8.07% baseline in the same direction as before. The relationship is stable across 
both the original and corrected flag logic — strengthens confidence this is a real 
signal (likely tied to housing stability/homeownership) rather than a counting 
artifact. Flag retained as a feature for Phase 3.

### 2. EXT_SOURCE Missingness Flags

EDA found that missingness in `EXT_SOURCE_1` and `EXT_SOURCE_3` carries predictive 
signal beyond their raw values (8.52% vs 7.50% default rate for EXT_SOURCE_1; 
9.31% vs 7.77% for EXT_SOURCE_3). Creating binary flags here, before any imputation, 
preserves this signal as a standalone feature. `EXT_SOURCE_2` excluded — only 0.21% 
missing, too rare to be a useful flag.

In [6]:
# Cell 5: Apply EXT_SOURCE missingness flags
from feature_engineering import add_ext_source_missing_flags

train_fe = add_ext_source_missing_flags(train_fe)
print(f"Shape after: {train_fe.shape}")    

EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
Shape after: (307511, 166)


**Result:** Flags created matching EDA findings exactly — `EXT_SOURCE_1_MISSING` 
56.38% (173,378), `EXT_SOURCE_3_MISSING` 19.83% (60,965). Shape: 95 → 97 columns.

### 3. Amount Transforms — Income and Credit

EDA found both `AMT_INCOME_TOTAL` and `AMT_CREDIT` heavily right-skewed, with 
log-transform producing a usable distribution for both. EDA also found and 
validated a single genuine outlier in `AMT_INCOME_TOTAL` (117,000,000 — isolated, 
not part of a legitimate heavy tail) which is capped at 10,000,000 before 
log-transforming. `AMT_CREDIT`'s max (4,050,000) was confirmed to be a legitimate 
loan product ceiling — no capping applied there.

In [7]:
# Cell 6: Apply amount transforms
from feature_engineering import transform_amounts

train_fe = transform_amounts(train_fe)
print(f"Shape after: {train_fe.shape}")

print(train_fe[['AMT_INCOME_TOTAL', 'AMT_INCOME_TOTAL_LOG', 'AMT_CREDIT', 'AMT_CREDIT_LOG']].describe())

AMT_INCOME_TOTAL: capped 0 rows at 10,000,000, log-transformed
AMT_CREDIT: log-transformed (no capping - ceiling confirmed legitimate in EDA)
Shape after: (307511, 166)
       AMT_INCOME_TOTAL  AMT_INCOME_TOTAL_LOG    AMT_CREDIT  AMT_CREDIT_LOG
count      3.075110e+05         307511.000000  3.075110e+05   307511.000000
mean       1.684126e+05             11.909234  5.990260e+05       13.070108
std        1.056929e+05              0.488791  4.024908e+05        0.715193
min        2.565000e+04             10.152338  4.500000e+04       10.714440
25%        1.125000e+05             11.630717  2.700000e+05       12.506181
50%        1.471500e+05             11.899215  5.135310e+05       13.149068
75%        2.025000e+05             12.218500  8.086500e+05       13.603123
max        1.000000e+07             16.118096  4.050000e+06       15.214228


**Result:** 3 rows capped in `AMT_INCOME_TOTAL` (matching EDA exactly), both 
columns log-transformed. Raw values preserved as `_RAW` columns for reference. 
Shape: 97 → 101 columns.

**Note:** hit a `NameError: name 'np' is not defined` on first run — `numpy` 
import was missing from the top of `feature_engineering.py` (notebook-level 
imports don't carry into imported modules). Fixed by adding `import numpy as np` 
directly in the file.

In [8]:
# Run this right after Cell 6 (transform_amounts), before Cell 7
print("OCCUPATION_TYPE present right after Cell 6:", 'OCCUPATION_TYPE' in train_fe.columns)
print("ORGANIZATION_TYPE present right after Cell 6:", 'ORGANIZATION_TYPE' in train_fe.columns)

OCCUPATION_TYPE present right after Cell 6: False
ORGANIZATION_TYPE present right after Cell 6: False


In [9]:
# Cell 8: Check OCCUPATION_TYPE missingness in the pre-encoding dataframe
print(f"Missing OCCUPATION_TYPE in train_raw: {train_raw['OCCUPATION_TYPE'].isnull().sum()}")
print(f"As percentage: {train_raw['OCCUPATION_TYPE'].isnull().sum() / len(train_raw) * 100:.2f}%")

Missing OCCUPATION_TYPE in train_raw: 96391
As percentage: 31.35%


In [10]:
# Cell 9 (corrected): Check overlap using DAYS_EMPLOYED placeholder directly
occ_missing = train_raw['OCCUPATION_TYPE'].isnull()
not_employed = (train_raw['DAYS_EMPLOYED'] == 365243)

overlap = pd.crosstab(occ_missing, not_employed)
overlap.index = ['Occupation present', 'Occupation missing']
overlap.columns = ['Employed', 'Not employed (365243)']
print(overlap)

                    Employed  Not employed (365243)
Occupation present    211118                      2
Occupation missing     41019                  55372


In [11]:
# Diagnostic: what's actually in encoding_maps right now?
print(type(encoding_maps))
print(encoding_maps.keys() if isinstance(encoding_maps, dict) else "not a dict")

<class 'dict'>
dict_keys(['ORGANIZATION_TYPE', 'OCCUPATION_TYPE'])


In [12]:
# Verify state is now consistent
print(f"train_fe shape: {train_fe.shape}")
print(f"encoding_maps keys: {list(encoding_maps.keys())}")
print(f"encoding_maps['OCCUPATION_TYPE'] has {len(encoding_maps.get('OCCUPATION_TYPE', {}))} categories")

train_fe shape: (307511, 166)
encoding_maps keys: ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
encoding_maps['OCCUPATION_TYPE'] has 19 categories


In [13]:
# Diagnostic: what's actually in train_fe right now?
print(f"Shape: {train_fe.shape}")
print(f"Has ORGANIZATION_TYPE column: {'ORGANIZATION_TYPE' in train_fe.columns}")
print(f"Has OCCUPATION_TYPE column: {'OCCUPATION_TYPE' in train_fe.columns}")
print(f"Has BUREAU_RECORD_COUNT: {'BUREAU_RECORD_COUNT' in train_fe.columns}")
print(f"Has INSTAL_RECORD_COUNT: {'INSTAL_RECORD_COUNT' in train_fe.columns}")

Shape: (307511, 166)
Has ORGANIZATION_TYPE column: False
Has OCCUPATION_TYPE column: False
Has BUREAU_RECORD_COUNT: False
Has INSTAL_RECORD_COUNT: False


In [14]:
# Cell 10: Check what the 'Missing' category learned for OCCUPATION_TYPE
print(encoding_maps['OCCUPATION_TYPE'].sort_values())

OCCUPATION_TYPE
Accountants              0.048303
High skill tech staff    0.061599
Managers                 0.062140
Core staff               0.063040
HR staff                 0.063943
IT staff                 0.064639
Missing                  0.065131
Private service staff    0.065988
Medicine staff           0.067002
Secretaries              0.070498
Realty agents            0.078562
Cleaning staff           0.096067
Sales staff              0.096318
Cooking staff            0.104440
Laborers                 0.105788
Security staff           0.107424
Waiters/barmen staff     0.112760
Drivers                  0.113261
Low-skill Laborers       0.171524
Name: TARGET, dtype: float64


**Verification:** the learned 'Missing' encoding for OCCUPATION_TYPE = 0.0651 
(6.51% default rate) — placing it in the lower-risk half of the distribution, 
between IT staff (6.46%) and Private service staff (6.60%), consistent with the 
pensioner-dominated population it represents. This confirms the fix was not 
cosmetic: the previous buggy fallback (generic global mean, 0.0852) would have 
meaningfully mis-encoded this group's true (lower) risk for 31% of the dataset. 

Broader pattern: occupation default rates form a clear socioeconomic gradient — 
white-collar/skilled roles cluster at 5-7% (Accountants, Managers, IT staff), 
manual/physical labor roles cluster at 10-17% (Low-skill Laborers, Drivers, 
Waiters) — consistent with the education and region-tier gradients found in EDA.

In [15]:
# Cell 11: Add ratio and interaction features
from feature_engineering import add_ratio_features

train_fe = add_ratio_features(train_fe)
print(f"\nShape after: {train_fe.shape}")

print(train_fe[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_GOODS_RATIO',
                 'ANNUITY_CREDIT_RATIO', 'AGE_YEARS', 'EMPLOYED_YEARS', 
                 'EMPLOYED_AGE_RATIO', 'INCOME_PER_PERSON']].describe())

Created 8 ratio/interaction features
Checking for inf/extreme values:
  ANNUITY_INCOME_RATIO: 0 inf, 12 null
  CREDIT_GOODS_RATIO: 0 inf, 278 null
  ANNUITY_CREDIT_RATIO: 0 inf, 12 null
  EMPLOYED_YEARS: 0 inf, 55374 null
  EMPLOYED_AGE_RATIO: 0 inf, 55374 null
  INCOME_PER_PERSON: 0 inf, 2 null

Shape after: (307511, 174)
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  CREDIT_GOODS_RATIO  ANNUITY_CREDIT_RATIO      AGE_YEARS  EMPLOYED_YEARS  EMPLOYED_AGE_RATIO  INCOME_PER_PERSON
count        307511.000000         307499.000000       307233.000000         307499.000000  307511.000000   252137.000000       252137.000000       3.075090e+05
mean              3.957571              0.180930            1.122995              0.053695      43.906900        6.527500            0.156861       9.297770e+04
std               2.689728              0.094574            0.124045              0.022481      11.947950        6.402081            0.133549       7.264940e+04
min               0.056249     

In [16]:
# Cell 12: Investigate the CREDIT_INCOME_RATIO outlier
extreme_ratio = train_fe.nlargest(5, 'CREDIT_INCOME_RATIO')[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CREDIT_INCOME_RATIO']]
print(extreme_ratio)

# Check the EMPLOYED_YEARS near-zero minimum
print(f"\nRows with EMPLOYED_YEARS very close to 0:")
print(train_fe[train_fe['EMPLOYED_YEARS'] < 0.01][['EMPLOYED_YEARS']].describe())

        AMT_INCOME_TOTAL  AMT_CREDIT  CREDIT_INCOME_RATIO
20727            25650.0   2173500.0            84.736842
35791            45000.0   2215224.0            49.227200
226137           45000.0   1800000.0            40.000000
255247           58500.0   2146500.0            36.692308
158077           40500.0   1436850.0            35.477778

Rows with EMPLOYED_YEARS very close to 0:
       EMPLOYED_YEARS
count        8.000000
mean         0.004791
std          0.003509
min         -0.000000
25%          0.002053
50%          0.005476
75%          0.008214
max          0.008214


### 5. Ratio and Interaction Features

Created 8 features contextualizing raw amounts relative to the applicant's own 
profile: CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO, CREDIT_GOODS_RATIO, 
ANNUITY_CREDIT_RATIO, AGE_YEARS, EMPLOYED_YEARS, EMPLOYED_AGE_RATIO, INCOME_PER_PERSON.

**Null counts confirmed expected:** EMPLOYED_YEARS/EMPLOYED_AGE_RATIO nulls (55,374) 
match IS_NOT_EMPLOYED exactly — correct, since pensioners have no employment duration. 
ANNUITY_*/CREDIT_GOODS_RATIO/INCOME_PER_PERSON nulls are small (≤278) and trace to 
pre-existing missingness in AMT_ANNUITY, AMT_GOODS_PRICE, CNT_FAM_MEMBERS — negligible.

**Investigated CREDIT_INCOME_RATIO max (84.7):** confirmed driven by genuinely low 
income (25,650 — the dataset's actual minimum) paired with a plausible loan amount 
(2,173,500, well below the 4,050,000 product ceiling) — not a data error like the 
earlier AMT_INCOME_TOTAL outlier. This is a real, informative high-risk signal. 
No capping applied; tree-based models handle this skew natively.

**Investigated EMPLOYED_YEARS near-zero minimum:** confirmed floating-point noise 
from DAYS_EMPLOYED values very close to 0 (8 rows, all under 0.0083 years/~3 days) 
— applicants who started employment essentially on application day. Not a data issue.

In [17]:
# Cell 13: Load bureau.csv and inspect structure first
bureau = pd.read_csv('../data/raw/home-credit-default-risk/bureau.csv')
print(f"Shape: {bureau.shape}")
print(f"Unique SK_ID_CURR: {bureau['SK_ID_CURR'].nunique()}")
print(f"Rows per applicant - describe:")
print(bureau.groupby('SK_ID_CURR').size().describe())
print(f"\nColumns:\n{bureau.columns.tolist()}")

Shape: (1716428, 17)
Unique SK_ID_CURR: 305811
Rows per applicant - describe:
count    305811.000000
mean          5.612709
std           4.430354
min           1.000000
25%           2.000000
50%           4.000000
75%           8.000000
max         116.000000
dtype: float64

Columns:
['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


In [18]:
# Cell 14: Check CREDIT_ACTIVE distribution before designing the aggregation
print(bureau['CREDIT_ACTIVE'].value_counts())

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64


In [19]:
# Cell 15: Aggregate and merge bureau.csv
from feature_engineering import aggregate_bureau, merge_bureau_features

bureau_agg = aggregate_bureau(bureau)
train_fe = merge_bureau_features(train_fe, bureau_agg)

print(f"\nShape after: {train_fe.shape}")

Aggregated bureau.csv: 305811 applicants -> 305811 rows, 45 columns
Merged bureau features: 45 new columns
Applicants with NO bureau record: 44020 (14.31%)

Shape after: (307511, 219)


In [20]:
# Cell 16: Diagnose the bureau merge count discrepancy
print(f"Expected applicants without bureau record: {307511 - 305811}")
print(f"Actual count from HAS_BUREAU_RECORD: {(train_fe['HAS_BUREAU_RECORD']==0).sum()}")

# Check if BUREAU_RECORD_COUNT actually has the expected number of nulls
print(f"\nBUREAU_RECORD_COUNT nulls: {train_fe['BUREAU_RECORD_COUNT'].isnull().sum()}")

# Check column name collisions - did 'BUREAU_RECORD_COUNT' get suffixed during merge?
bureau_related_cols = [c for c in train_fe.columns if 'RECORD_COUNT' in c or 'BUREAU_RECORD' in c]
print(f"\nColumns matching 'RECORD_COUNT' or 'BUREAU_RECORD': {bureau_related_cols}")

Expected applicants without bureau record: 1700
Actual count from HAS_BUREAU_RECORD: 44020

BUREAU_RECORD_COUNT nulls: 44020

Columns matching 'RECORD_COUNT' or 'BUREAU_RECORD': ['BUREAU_RECORD_COUNT', 'HAS_BUREAU_RECORD']


In [21]:
# Cell 17: Check SK_ID_CURR overlap directly between train and bureau_agg
train_ids = set(train_raw['SK_ID_CURR'])
bureau_ids = set(bureau_agg['SK_ID_CURR'])

print(f"train_raw unique IDs: {len(train_ids)}")
print(f"bureau_agg unique IDs: {len(bureau_ids)}")
print(f"IDs in train but NOT in bureau_agg: {len(train_ids - bureau_ids)}")
print(f"IDs in bureau_agg but NOT in train: {len(bureau_ids - train_ids)}")

# Check dtypes - a dtype mismatch (e.g. int64 vs object) would break the merge silently
print(f"\ntrain_raw SK_ID_CURR dtype: {train_raw['SK_ID_CURR'].dtype}")
print(f"bureau_agg SK_ID_CURR dtype: {bureau_agg['SK_ID_CURR'].dtype}")

train_raw unique IDs: 307511
bureau_agg unique IDs: 305811
IDs in train but NOT in bureau_agg: 44020
IDs in bureau_agg but NOT in train: 42320

train_raw SK_ID_CURR dtype: int64
bureau_agg SK_ID_CURR dtype: int64


### 6. Bureau Table Aggregation

Aggregated `bureau.csv` (1,716,428 rows, multiple bureau credit-lines per 
applicant) into one row per `SK_ID_CURR`: CREDIT_ACTIVE and CREDIT_TYPE pivoted 
into count columns (status/type carries distinct risk meaning, not safely 
averaged), numeric fields aggregated via min/max/mean/sum as appropriate.

**Reasoning error caught and corrected:** initially predicted only ~1,700 
applicants (307,511 - 305,811) would have no bureau record, based on the naive 
assumption that all 305,811 unique IDs in `bureau.csv` belong to the training 
set. Direct ID-overlap check revealed `bureau.csv` contains records for **both** 
train and test applicants combined (42,320 of its 305,811 IDs belong to the 
test set, not train). 

**Corrected finding: 44,020 applicants (14.31% of training data) have no 
bureau record at all** — a substantially larger "credit-invisible-within-bureau" 
population than initially assumed, and a stronger empirical anchor for this 
project's underbanked-population framing than the original (incorrect) estimate.

This was a reasoning error, not a code bug — the aggregation and merge logic 
were correct throughout; the mistake was in the train/test composition 
assumption used to sanity-check the result. Important methodological lesson: 
verify ID-set overlaps directly rather than inferring them from row-count 
arithmetic alone.

In [22]:
# Cell 18: Load previous_application.csv and inspect structure
prev_app = pd.read_csv('../data/raw/home-credit-default-risk/previous_application.csv')
print(f"Shape: {prev_app.shape}")
print(f"Unique SK_ID_CURR: {prev_app['SK_ID_CURR'].nunique()}")
print(f"Rows per applicant - describe:")
print(prev_app.groupby('SK_ID_CURR').size().describe())
print(f"\nColumns:\n{prev_app.columns.tolist()}")   

Shape: (1670214, 37)
Unique SK_ID_CURR: 338857
Rows per applicant - describe:
count    338857.000000
mean          4.928964
std           4.220716
min           1.000000
25%           2.000000
50%           4.000000
75%           7.000000
max          77.000000
dtype: float64

Columns:
['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST

In [23]:
# Cell 19: Check NAME_CONTRACT_STATUS distribution
print(prev_app['NAME_CONTRACT_STATUS'].value_counts())

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64


In [24]:
# Cell 20: Check missingness on the rate columns before deciding whether to include them
for col in ['RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'RATE_DOWN_PAYMENT']:
    pct = prev_app[col].isnull().sum() / len(prev_app) * 100
    print(f"{col}: {pct:.2f}% missing")

RATE_INTEREST_PRIMARY: 99.64% missing
RATE_INTEREST_PRIVILEGED: 99.64% missing
RATE_DOWN_PAYMENT: 53.64% missing


In [25]:
# Cell 21: Check ID overlap directly first (lesson from bureau.csv)
train_ids = set(train_raw['SK_ID_CURR'])
prev_ids = set(prev_app['SK_ID_CURR'])
print(f"train_raw IDs: {len(train_ids)}")
print(f"prev_app unique IDs: {len(prev_ids)}")
print(f"IDs in train but NOT in prev_app: {len(train_ids - prev_ids)}")
print(f"IDs in prev_app but NOT in train: {len(prev_ids - train_ids)}")

train_raw IDs: 307511
prev_app unique IDs: 338857
IDs in train but NOT in prev_app: 16454
IDs in prev_app but NOT in train: 47800


In [26]:
# Cell 22: Aggregate and merge previous_application.csv
from feature_engineering import aggregate_previous_application, merge_previous_application_features

prev_agg = aggregate_previous_application(prev_app)
train_fe = merge_previous_application_features(train_fe, prev_agg)

print(f"\nShape after: {train_fe.shape}")

Aggregated previous_application.csv: 338857 applicants -> 338857 rows, 31 columns
Merged previous_application features: 31 new columns
Applicants with NO previous Home Credit application: 16454 (5.35%)

Shape after: (307511, 250)


### 7. Previous Application Table Aggregation

Aggregated `previous_application.csv` (1,670,214 rows) into one row per 
`SK_ID_CURR`. NAME_CONTRACT_STATUS pivoted into count columns (Approved/Refused/
Canceled/Unused offer carry distinct risk meaning); added a directly interpretable 
PREV_REFUSAL_RATE feature. RATE_INTEREST_PRIMARY/RATE_INTEREST_PRIVILEGED excluded 
- confirmed 99.64% missing via direct check, essentially no signal.

**Applied lesson from bureau.csv:** verified SK_ID_CURR overlap directly via set 
operations *before* merging, rather than inferring missing-record counts from row 
totals. This confirmed cleanly: 16,454 applicants (5.35%) have no previous Home 
Credit application — exactly matching the merge result, no discrepancy this time.

**Finding:** 94.65% of applicants have at least one prior Home Credit application 
- a high repeat-customer rate, likely reflecting Home Credit's practice of 
campaigning to its existing customer base. The genuinely "blind" population (no 
bureau record AND no prior Home Credit application) is a specific, identifiable 
subgroup worth examining directly when characterizing this project's target 
underbanked population.

In [27]:
# Cell 23: Load installments_payments.csv and inspect structure
installments = pd.read_csv('../data/raw/home-credit-default-risk/installments_payments.csv')
print(f"Shape: {installments.shape}")
print(f"Unique SK_ID_CURR: {installments['SK_ID_CURR'].nunique()}")
print(f"Rows per applicant - describe:")
print(installments.groupby('SK_ID_CURR').size().describe())
print(f"\nColumns:\n{installments.columns.tolist()}")

Shape: (13605401, 8)
Unique SK_ID_CURR: 339587
Rows per applicant - describe:
count    339587.000000
mean         40.064552
std          41.053343
min           1.000000
25%          12.000000
50%          25.000000
75%          51.000000
max         372.000000
dtype: float64

Columns:
['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']


In [28]:
# Cell 24: Quick check on the payment timing/amount gap before designing aggregation
sample = installments[['DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']].describe()
print(sample)

# Check for nulls in DAYS_ENTRY_PAYMENT / AMT_PAYMENT specifically (missed payments would show here)
print(f"\nDAYS_ENTRY_PAYMENT nulls: {installments['DAYS_ENTRY_PAYMENT'].isnull().sum()}")
print(f"AMT_PAYMENT nulls: {installments['AMT_PAYMENT'].isnull().sum()}")


       DAYS_INSTALMENT  DAYS_ENTRY_PAYMENT  AMT_INSTALMENT   AMT_PAYMENT
count     1.360540e+07        1.360250e+07    1.360540e+07  1.360250e+07
mean     -1.042270e+03       -1.051114e+03    1.705091e+04  1.723822e+04
std       8.009463e+02        8.005859e+02    5.057025e+04  5.473578e+04
min      -2.922000e+03       -4.921000e+03    0.000000e+00  0.000000e+00
25%      -1.654000e+03       -1.662000e+03    4.226085e+03  3.398265e+03
50%      -8.180000e+02       -8.270000e+02    8.884080e+03  8.125515e+03
75%      -3.610000e+02       -3.700000e+02    1.671021e+04  1.610842e+04
max      -1.000000e+00       -1.000000e+00    3.771488e+06  3.771488e+06

DAYS_ENTRY_PAYMENT nulls: 2905
AMT_PAYMENT nulls: 2905


In [29]:
# Cell 25: Check ID overlap, then aggregate and merge
train_ids = set(train_raw['SK_ID_CURR'])
instal_ids = set(installments['SK_ID_CURR'])
print(f"IDs in train but NOT in installments: {len(train_ids - instal_ids)}")

IDs in train but NOT in installments: 15868


In [30]:
# Cell 26: Aggregate and merge installments_payments.csv (may take a moment - 13.6M rows)
from feature_engineering import aggregate_installments, merge_installments_features

instal_agg = aggregate_installments(installments)
train_fe = merge_installments_features(train_fe, instal_agg)

print(f"\nShape after: {train_fe.shape}")

Aggregated installments_payments.csv: 339587 applicants -> 339587 rows, 16 columns
Merged installments features: 16 new columns
Applicants with NO installment history: 15868 (5.16%)

Shape after: (307511, 266)


In [31]:
# Diagnostic: check what columns actually came out of the new aggregation
print(instal_agg.columns.tolist())

['SK_ID_CURR', 'INSTAL_DAYS_LATE_MAX', 'INSTAL_DAYS_LATE_MEAN', 'INSTAL_AMT_SHORTFALL_MAX', 'INSTAL_AMT_SHORTFALL_MEAN', 'INSTAL_AMT_SHORTFALL_SUM', 'INSTAL_MISSED_PAYMENT_SUM', 'INSTAL_AMT_INSTALMENT_MAX', 'INSTAL_AMT_INSTALMENT_MEAN', 'INSTAL_AMT_INSTALMENT_SUM', 'INSTAL_AMT_PAYMENT_MAX', 'INSTAL_AMT_PAYMENT_MEAN', 'INSTAL_AMT_PAYMENT_SUM', 'INSTAL_NUM_INSTALMENT_NUMBER_MAX', 'INSTAL_RECORD_COUNT', 'INSTAL_LATE_PAYMENT_RATE']


In [32]:
# Diagnostic: check if train_fe already has installment columns from a previous run
instal_cols_in_train_fe = [c for c in train_fe.columns if 'INSTAL' in c]
print(f"INSTAL-related columns already in train_fe: {instal_cols_in_train_fe}")
print(f"\ntrain_fe shape right now: {train_fe.shape}")

INSTAL-related columns already in train_fe: ['INSTAL_DAYS_LATE_MAX', 'INSTAL_DAYS_LATE_MEAN', 'INSTAL_AMT_SHORTFALL_MAX', 'INSTAL_AMT_SHORTFALL_MEAN', 'INSTAL_AMT_SHORTFALL_SUM', 'INSTAL_MISSED_PAYMENT_SUM', 'INSTAL_AMT_INSTALMENT_MAX', 'INSTAL_AMT_INSTALMENT_MEAN', 'INSTAL_AMT_INSTALMENT_SUM', 'INSTAL_AMT_PAYMENT_MAX', 'INSTAL_AMT_PAYMENT_MEAN', 'INSTAL_AMT_PAYMENT_SUM', 'INSTAL_NUM_INSTALMENT_NUMBER_MAX', 'INSTAL_RECORD_COUNT', 'INSTAL_LATE_PAYMENT_RATE', 'HAS_INSTALLMENT_HISTORY']

train_fe shape right now: (307511, 266)


In [33]:
# Final check after clean restart
print(f"Shape: {train_fe.shape}")
print(f"INSTAL_RECORD_COUNT present: {'INSTAL_RECORD_COUNT' in train_fe.columns}")
print(f"No duplicate suffixed columns: {[c for c in train_fe.columns if c.endswith('_x') or c.endswith('_y')]}")

Shape: (307511, 266)
INSTAL_RECORD_COUNT present: True
No duplicate suffixed columns: []


**Process note:** hit a duplicate-merge error when re-running a single cell 
(Cell 26) in isolation after a code fix, rather than restarting the kernel and 
re-running the full pipeline. `train_fe` already contained installment columns 
from the prior successful run; re-merging produced `_x`/`_y` suffixed duplicates, 
breaking the `INSTAL_RECORD_COUNT` reference downstream. Not a logic bug - a 
notebook state hygiene issue. **Going forward: after any fix to a function in 
feature_engineering.py, restart the kernel and re-run the full pipeline from 
Cell 2, rather than re-running the affected cell standalone.**

In [ ]:
print(f"train_fe shape: {train_fe.shape}")
print(f"encoding_maps keys: {list(encoding_maps.keys())}")

train_fe shape: (307511, 266)
encoding_maps keys: ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE']
